Load Dataset - Enron Scandal 

In [0]:
dbutils.fs.ls("/FileStore/tables/")

Out[1]: [FileInfo(path='dbfs:/FileStore/tables/BDTT_Assignment_1_Enron/', name='BDTT_Assignment_1_Enron/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/FileStore/tables/BDTT_Assignment_1_Enron.zip', name='BDTT_Assignment_1_Enron.zip', size=375294957, modificationTime=1739368129000),
 FileInfo(path='dbfs:/FileStore/tables/assignment/', name='assignment/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/FileStore/tables/test.json', name='test.json', size=17958, modificationTime=1737554337000),
 FileInfo(path='dbfs:/FileStore/tables/webpage/', name='webpage/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/FileStore/tables/webpage.zip', name='webpage.zip', size=1582, modificationTime=1738765332000),
 FileInfo(path='dbfs:/FileStore/tables/webpage_files_all/', name='webpage_files_all/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/FileStore/tables/webpage_files_jpg/', name='webpage_files_jpg/', size=0, modificationTime=0)]

Copy zip folder to DBFS

In [0]:
dbutils.fs.cp("/FileStore/tables/BDTT_Assignment_1_Enron.zip", "file:/tmp/")

Out[2]: True

In [0]:
%sh
ls /tmp/

BDTT_Assignment_1_Enron.zip
Rserv
RtmpW4uEQb
chauffeur-daemon-params
chauffeur-daemon.pid
chauffeur-env.sh
custom-spark.conf
driver-daemon-params
driver-daemon.pid
driver-env.sh
hsperfdata_root
systemd-private-24672987137549118db34a9964d57504-apache2.service-P4LyKg
systemd-private-24672987137549118db34a9964d57504-ntp.service-Rxpnwf
systemd-private-24672987137549118db34a9964d57504-systemd-logind.service-4VjGYh
systemd-private-24672987137549118db34a9964d57504-systemd-resolved.service-6TLE9g
tmp.jXCOJ2roDb


Unzip dataset

In [0]:
%sh
unzip -d /tmp/ /tmp/BDTT_Assignment_1_Enron.zip

Archive:  /tmp/BDTT_Assignment_1_Enron.zip
  inflating: /tmp/emails.csv         


In [0]:
%sh
ls /tmp/emails.csv

/tmp/emails.csv


Create folder in DBFS and move the files from Local node to DBFS

In [0]:
dbutils.fs.mkdirs("FileStore/tables/BDTT_Assignment_1_Enron")

Out[6]: True

In [0]:
dbutils.fs.mv("file:/tmp/emails.csv", "/FileStore/tables/BDTT_Assignment_1_Enron", True)

Out[7]: True

In [0]:
dbutils.fs.ls("FileStore/tables/BDTT_Assignment_1_Enron/")

Out[8]: [FileInfo(path='dbfs:/FileStore/tables/BDTT_Assignment_1_Enron/emails.csv', name='emails.csv', size=1426122219, modificationTime=1741514965000)]

Display Dataset Head - 500 bytes

In [0]:
dbutils.fs.head("FileStore/tables/BDTT_Assignment_1_Enron/emails.csv", 600)

[Truncated to first 600 bytes]
Out[9]: '"file","message"\n"allen-p/_sent_mail/1.","Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>\nDate: Mon, 14 May 2001 16:39:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: tim.belden@enron.com\nSubject: \nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-To: Tim Belden <Tim Belden/Enron@EnronXGate>\nX-cc: \nX-bcc: \nX-Folder: \\Phillip_Allen_Jan2002_1\\Allen, Phillip K.\\\'Sent Mail\nX-Origin: Allen-P\nX-FileName: pallen (Non-Privileged).pst\n\nHere is our forecast\n\n "\n"allen-p/_sent_mail/10.","Message-ID: <15464986.1075855378456.Ja'

In [0]:
for line in dbutils.fs.head('FileStore/tables/BDTT_Assignment_1_Enron/emails.csv', 600).splitlines():
    print(line)

[Truncated to first 600 bytes]
"file","message"
"allen-p/_sent_mail/1.","Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>
Date: Mon, 14 May 2001 16:39:00 -0700 (PDT)
From: phillip.allen@enron.com
To: tim.belden@enron.com
Subject: 
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: Tim Belden <Tim Belden/Enron@EnronXGate>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).pst

Here is our forecast

 "
"allen-p/_sent_mail/10.","Message-ID: <15464986.1075855378456.Ja


Reading Dataset as per Assessment Brief

In [0]:
df = spark.read.csv(
"/FileStore/tables/BDTT_Assignment_1_Enron/emails.csv",
header=True, # Use the first row as the header
inferSchema=True, # Infer data types
quote='"', # Define the quote character
escape='"', # Escape quotes inside quoted fields
multiLine=True # Enable multiline support
)

In [0]:
 df.head()

Out[12]: Row(file='allen-p/_sent_mail/1.', message="Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>\nDate: Mon, 14 May 2001 16:39:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: tim.belden@enron.com\nSubject: \nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-To: Tim Belden <Tim Belden/Enron@EnronXGate>\nX-cc: \nX-bcc: \nX-Folder: \\Phillip_Allen_Jan2002_1\\Allen, Phillip K.\\'Sent Mail\nX-Origin: Allen-P\nX-FileName: pallen (Non-Privileged).pst\n\nHere is our forecast\n\n ")

There are Two Columns File and Message

In [0]:
df.columns

Out[13]: ['file', 'message']

In [0]:
rdd = df.select("message").rdd.map(lambda row: row["message"] if row["message"] else "")


In [0]:
rdd.take(2)


Out[15]: ["Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>\nDate: Mon, 14 May 2001 16:39:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: tim.belden@enron.com\nSubject: \nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-To: Tim Belden <Tim Belden/Enron@EnronXGate>\nX-cc: \nX-bcc: \nX-Folder: \\Phillip_Allen_Jan2002_1\\Allen, Phillip K.\\'Sent Mail\nX-Origin: Allen-P\nX-FileName: pallen (Non-Privileged).pst\n\nHere is our forecast\n\n ",
 "Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>\nDate: Fri, 4 May 2001 13:51:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: john.lavorato@enron.com\nSubject: Re:\nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-To: John J Lavorato <John J Lavorato/ENRON@enronXgate@ENRON>\nX-cc: \nX-bcc: \nX-Folder: \\Phillip_Allen_Jan2002_1\\Allen, Phillip K.\\'Sent Mail\nX-Origin: Allen-P\nX-Fil

In [0]:
import re
from pyspark.sql import functions as F

# Email header patterns
patterns = {
    "Message-ID": r"Message-ID: <([^>]+)>",
    "Date": r"Date: (.*?)(?:\n|$)",
    "From": r"From: (.*?)(?:\n|$)",
    "To": r"To: (.*?)(?:\n|$)",
    "Subject": r"Subject: (.*?)(?:\n|$)",
    "Mime-Version": r"Mime-Version: (.*?)(?:\n|$)",
    "Content-Type": r"Content-Type: (.*?)(?:\n|$)",
    "Content-Transfer-Encoding": r"Content-Transfer-Encoding: (.*?)(?:\n|$)",
    "X-From": r"X-From: (.*?)(?:\n|$)",
    "X-To": r"X-To: (.*?)(?:\n|$)",
    "X-cc": r"X-cc: (.*?)(?:\n|$)",
    "X-bcc": r"X-bcc: (.*?)(?:\n|$)",
    "X-Folder": r"X-Folder: (.*?)(?:\n|$)",
    "X-Origin": r"X-Origin: (.*?)(?:\n|$)",
    "X-FileName": r"X-FileName: (.*?)(?:\n|$)",
}

# Parsing each email
parsed_rdd = rdd.map(lambda email_data: {
    key: re.search(pattern, email_data).group(1).strip() if re.search(pattern, email_data) else ""
    for key, pattern in patterns.items()
})

# Example: Collecting results (you can adjust how you want to process it, such as converting to a DataFrame)
parsed_email_list = parsed_rdd.collect()


In [0]:
# Convert parsed_rdd to DataFrame
df = spark.createDataFrame(parsed_rdd)


Exploratory Data Analysis

In [0]:
# 1. Check column names
print("Columns in the DataFrame:", df.columns)

# 2. Show total number of rows
total_entries = df.count()
print(f"Total entries in the DataFrame: {total_entries}")

# 3. Display the first 5 rows
df.show(5)

# 4. Check for null or missing values in each column
print("Null values in each column:")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# 5. Summary statistics for numeric columns
print("Summary statistics for numeric columns:")
df.describe().show()

# 6. Check data types of each column
print("Data types of each column:", df.dtypes)

Columns in the DataFrame: ['Content-Transfer-Encoding', 'Content-Type', 'Date', 'From', 'Message-ID', 'Mime-Version', 'Subject', 'To', 'X-FileName', 'X-Folder', 'X-From', 'X-Origin', 'X-To', 'X-bcc', 'X-cc']
Total entries in the DataFrame: 517401
+-------------------------+--------------------+--------------------+--------------------+--------------------+------------+---------+--------------------+--------------------+--------------------+---------------+--------+--------------------+-----+----+
|Content-Transfer-Encoding|        Content-Type|                Date|                From|          Message-ID|Mime-Version|  Subject|                  To|          X-FileName|            X-Folder|         X-From|X-Origin|                X-To|X-bcc|X-cc|
+-------------------------+--------------------+--------------------+--------------------+--------------------+------------+---------+--------------------+--------------------+--------------------+---------------+--------+--------------------+

Group (A) - Question 3:  What is the number of emails with a subject line (i.e., the 'Subject' field is not missing or empty)?

In [0]:
from pyspark.sql.functions import col

df.select(col("Message-ID"), col("Subject")).show(10, truncate=False)

+-------------------------------------------+-------------------------------------------------------+
|Message-ID                                 |Subject                                                |
+-------------------------------------------+-------------------------------------------------------+
|18782981.1075855378110.JavaMail.evans@thyme|                                                       |
|15464986.1075855378456.JavaMail.evans@thyme|Re:                                                    |
|24216240.1075855687451.JavaMail.evans@thyme|Re: test                                               |
|13505866.1075863688222.JavaMail.evans@thyme|                                                       |
|30922949.1075863688243.JavaMail.evans@thyme|Re: Hello                                              |
|30965995.1075863688265.JavaMail.evans@thyme|Re: Hello                                              |
|16254169.1075863688286.JavaMail.evans@thyme|                                     

As it can be seen through visualizing a sample of 20 emails subjects, there can be a few ways for a subject to empty:

- an empty subject, that has no charecter.
- a reply or forward of emails with no subjcet, this is were the subject line contains either "Re:" or "FW:", which means relpying or forwarding an email with no subject, for this reason we have counted these subjects as empty as well.

In [0]:
from pyspark.sql.functions import col, lower

# Filtering
count1 = df.filter(
   lower(col("Subject")) == ""
).count()

count2 = df.filter(
   lower(col("Subject")) == "re:"
).count()

count3 = df.filter(
   lower(col("Subject")) == "fw:"
).count()

count4 = df.filter(
   lower(col("Subject")) == "fwd:"
).count()

print("Count of empty Subjects ", count1)
print("Count of Subjects with re: ", count2)
print("Count of Subjects with fw: ", count3)
print("Count of Subjects with fwd: ", count4)

Count of empty Subjects  19187
Count of Subjects with re:  12788
Count of Subjects with fw:  944
Count of Subjects with fwd:  86


In [0]:
from pyspark.sql.functions import col, lower

# Filtering and counting rows
count = df.filter(
   ~ lower(col("Subject")).isin(["", "re:", "fw:", "fwd:"])
).count()

print(count)

484396
